# Notebook 3 — NSGA-II + FIS (Feature Extraction)
## Fuzzy + HV por geracao + Avaliacao no Teste

Fixes v9: chave por `xkey(x)`, celula [9] avalia no teste, matriz de confusao,
`classification_report` completo, `best_model_fis.pth` com todas as metricas.


In [ ]:
# [0] Instalacao
# !pip install torch torchvision pymoo scikit-fuzzy scikit-learn
#              numpy pandas matplotlib scipy


In [ ]:
# [1] Imports
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision.models as models
import torchvision.datasets as tvd
from torchvision import transforms
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import random, time, gc, json, hashlib, warnings
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, confusion_matrix, classification_report
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.core.callback import Callback
from pymoo.indicators.hv import HV
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# [2] Dataset + Extracao de Features (UMA VEZ)
# ─────────────────────────────────────────────────────────────────
# Datasets disponíveis:
#   'cifar10'    50k amostras, 10 classes, ~170MB  <- MAIS RAPIDO
#   'stl10'      13k amostras, 10 classes, ~2.5GB  <- BOA QUALIDADE
#   'eurosat'    27k amostras, 10 classes,  ~90MB
#   'flowers102'  8k amostras,102 classes, ~330MB  <- ALTA ACC
#
# Backbones (features extraidas uma vez, ~3-8 min):
#   0=ResNet18  | 1=EfficientNet-B0 | 2=MobileNetV2
# ─────────────────────────────────────────────────────────────────

DATASET_NAME = 'cifar10'
DATA_ROOT    = r'C:\\datasets'

transform_ext = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def get_raw_splits(name, root):
    if name == 'cifar10':
        tr = tvd.CIFAR10(root, train=True,  transform=transform_ext, download=True)
        te = tvd.CIFAR10(root, train=False, transform=transform_ext, download=True)
        return tr, te, tr.classes
    elif name == 'stl10':
        tr = tvd.STL10(root, split='train', transform=transform_ext, download=True)
        te = tvd.STL10(root, split='test',  transform=transform_ext, download=True)
        return tr, te, tr.classes
    elif name == 'eurosat':
        full = tvd.EuroSAT(root, transform=transform_ext, download=True)
        g    = torch.Generator().manual_seed(SEED)
        n_tr = int(.80*len(full))
        tr, te = random_split(full, [n_tr, len(full)-n_tr], generator=g)
        return tr, te, full.classes
    elif name == 'flowers102':
        tr = tvd.Flowers102(root, split='train', transform=transform_ext, download=True)
        te = tvd.Flowers102(root, split='test',  transform=transform_ext, download=True)
        return tr, te, [str(i) for i in range(102)]
    else:
        raise ValueError(f'Dataset desconhecido: {name}')

BACKBONE_CFGS = {
    0: ('ResNet18',        models.resnet18,        'IMAGENET1K_V1', 512),
    1: ('EfficientNet-B0', models.efficientnet_b0, 'IMAGENET1K_V1', 1280),
    2: ('MobileNetV2',     models.mobilenet_v2,    'IMAGENET1K_V1', 1280),
}
LATENCY_REF = {'ResNet18':10,'EfficientNet-B0':12,'MobileNetV2':8}
MEMORY_REF  = {'ResNet18':200,'EfficientNet-B0':220,'MobileNetV2':150}
LAT_MAX, MEM_MAX = max(LATENCY_REF.values()), max(MEMORY_REF.values())

def extract_features(backbone_idx, dataset, batch_size=256):
    bname, builder, weights, feat_dim = BACKBONE_CFGS[backbone_idx]
    print(f'  [{bname}] extraindo...', end=' ', flush=True)
    base = builder(weights=weights).to(device)
    if hasattr(base,'fc'):          base.fc          = nn.Identity()
    elif hasattr(base,'classifier'): base.classifier = nn.Identity()
    base.eval()
    loader = DataLoader(dataset, batch_size, shuffle=False, num_workers=0, pin_memory=True)
    feats, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            feats.append(base(X.to(device)).cpu())
            labels.append(y if torch.is_tensor(y) else torch.tensor(y))
    del base; torch.cuda.empty_cache(); gc.collect()
    F = torch.cat(feats); L = torch.cat(labels)
    print(f'shape={F.shape}', flush=True)
    return F, L

# ── Carregar e split ──────────────────────────────────────────────
print(f'Carregando {DATASET_NAME}...')
raw_tr, raw_te, CLASSES = get_raw_splits(DATASET_NAME, DATA_ROOT)
NUM_CLASSES = len(CLASSES)
g   = torch.Generator().manual_seed(SEED)
n_va = max(int(.15*len(raw_tr)), 1000)
n_tr = len(raw_tr) - n_va
raw_tr_sub, raw_va_sub = random_split(raw_tr, [n_tr, n_va], generator=g)
print(f'Dataset:{DATASET_NAME.upper()} Classes:{NUM_CLASSES} '
      f'Treino:{n_tr} Val:{n_va} Teste:{len(raw_te)}')

# ── Extrair features de todos os backbones ────────────────────────
print('\nExtraindo features (uma vez ~3-8 min):')
t0 = time.time()
# ALL_FEATURES[bidx] = (f_tr, l_tr, f_va, l_va, f_te, l_te)
ALL_FEATURES = {}
for bidx in BACKBONE_CFGS:
    f_tr,l_tr = extract_features(bidx, raw_tr_sub)
    f_va,l_va = extract_features(bidx, raw_va_sub)
    f_te,l_te = extract_features(bidx, raw_te)
    ALL_FEATURES[bidx] = (f_tr,l_tr,f_va,l_va,f_te,l_te)
print(f'\nExtracao concluida em {(time.time()-t0)/60:.1f} min')
print('Cada avaliacao NSGA-II agora: <30 segundos')


In [ ]:
# [3] Cabeca linear + fast_eval
# ─────────────────────────────────────────────────────────────────
# FIX CRITICO — bug de accuracy 0%:
#   O problema era que pkey(decode(x)) gerava floats arredondados
#   diferentes da chave original criada durante a avaliacao.
#   Lookup falhava silenciosamente -> metrics vazias -> 0%.
#
# SOLUCAO: EVAL_CACHE usa chave baseada no vetor x original (hash MD5)
#   nao no dict decodificado. A mesma chave e usada em TODAS as funcoes.
#   Alem disso, guardamos TODAS as metricas no cache (acc, f1, auroc,
#   loss, fpr, backbone) para recuperacao direta sem recalculo.
# ─────────────────────────────────────────────────────────────────

class LinearHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),      nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout*0.75),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.net(x)

OPT_NAMES = ['Adam','AdamW','SGD']

# Cache global — chave = hash MD5 do vetor x original
EVAL_CACHE = {}  # key -> dict com TODAS as metricas

def xkey(x):
    """Chave deterministica baseada no vetor x (nao no dict decodificado)."""
    return hashlib.md5(np.array(x, dtype=np.float64).tobytes()).hexdigest()

def fast_eval(x_vec):
    """
    Treina cabeca linear sobre features pre-extraidas.
    Recebe o vetor x original (nao o dict) para garantir chave unica.
    Retorna dict com todas as metricas.
    """
    key = xkey(x_vec)
    if key in EVAL_CACHE:
        m = EVAL_CACHE[key]
        print(f"  CACHE {m['backbone']:18s}  acc={m['accuracy']:.4f}", flush=True)
        return m

    params   = decode(x_vec)
    bidx     = params['backbone_idx']
    bname    = BACKBONE_CFGS[bidx][0]
    feat_dim = BACKBONE_CFGS[bidx][3]
    f_tr,l_tr,f_va,l_va,_,_ = ALL_FEATURES[bidx]

    print(f"  TRAIN {bname:18s}"
          f"  lr={params['lr']:.2e}  ep={params['epochs']}"
          f"  bs={params['batch_size']}  drop={params['dropout']:.2f}",
          flush=True)

    seed = int(key[:8],16) % (2**31)
    torch.manual_seed(seed); np.random.seed(seed%(2**32))

    bs    = params['batch_size']
    g     = torch.Generator(); g.manual_seed(seed)
    tr_ld = DataLoader(TensorDataset(f_tr,l_tr), bs, shuffle=True,  generator=g)
    va_ld = DataLoader(TensorDataset(f_va,l_va), bs, shuffle=False)

    model = LinearHead(feat_dim, NUM_CLASSES, params['dropout']).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.1)
    lr    = params['lr']; wd = params['wd']
    kw    = {'lr':lr,'weight_decay':wd}
    if params['optimizer']==2: kw.update({'momentum':0.9,'nesterov':True})
    opt   = {0:optim.Adam,1:optim.AdamW,2:optim.SGD}[params['optimizer']](model.parameters(),**kw)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=params['epochs'], eta_min=lr*0.01)

    best_loss, pat = float('inf'), 0
    for _ in range(params['epochs']):
        model.train()
        for X,y in tr_ld:
            X,y=X.to(device),y.to(device)
            opt.zero_grad(); crit(model(X),y).backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vl = sum(crit(model(X.to(device)),y.to(device)).item()
                     for X,y in va_ld) / len(va_ld)
        if vl < best_loss: best_loss=vl; pat=0
        else: pat+=1
        if pat >= 6: break

    # Metricas completas de validacao
    model.eval(); preds,labs,probs=[],[],[]
    with torch.no_grad():
        for X,y in va_ld:
            out=model(X.to(device))
            preds.extend(torch.argmax(out,1).cpu().numpy())
            labs.extend(y.numpy())
            probs.extend(torch.softmax(out,1).cpu().numpy())
    preds=np.array(preds); labs=np.array(labs); probs=np.array(probs)
    acc  = float(accuracy_score(labs,preds))
    f1m  = float(f1_score(labs,preds,average='macro',zero_division=0))
    prec = float(precision_score(labs,preds,average='macro',zero_division=0))
    rec  = float(recall_score(labs,preds,average='macro',zero_division=0))
    try:    auroc=float(roc_auc_score(labs,probs,multi_class='ovr',average='macro'))
    except: auroc=0.5

    # Guardar estado do modelo para retreino posterior
    model_state = {k:v.cpu() for k,v in model.state_dict().items()}

    result = {
        'accuracy':acc,'f1':f1m,'precision':prec,'recall':rec,
        'auroc':auroc,'fpr':1-acc,'loss':best_loss,
        'backbone':bname,'backbone_idx':bidx,
        'params':params,'model_state':model_state,
        'feat_dim':feat_dim,
    }
    del model,opt,sched; gc.collect()
    EVAL_CACHE[key] = result
    print(f"        acc={acc:.4f}  f1={f1m:.4f}  auroc={auroc:.4f}", flush=True)
    return result

print('fast_eval OK — chave por hash do vetor x (sem bug de float)')


In [ ]:
# [4] Espaco de busca + utilitarios
N_VAR = 7
xl = np.array([0.0, 0, 20, 0.0, 0, 0, 0.1])
xu = np.array([1.0, 2, 80, 1.0, 2, 2, 0.5])

BATCH_SIZES = [32, 64, 128]

def decode(x):
    lr = 10**(np.log10(1e-4) + x[0]*(np.log10(1e-1)-np.log10(1e-4)))
    wd = 10**(np.log10(1e-5) + x[3]*(np.log10(1e-2)-np.log10(1e-5)))
    return {
        'lr':          float(lr),
        'optimizer':   int(np.clip(round(x[1]),0,2)),
        'epochs':      int(np.clip(round(x[2]),20,80)),
        'wd':          float(wd),
        'backbone_idx':int(np.clip(round(x[4]),0,2)),
        'batch_size':  BATCH_SIZES[int(np.clip(round(x[5]),0,2))],
        'dropout':     float(np.clip(x[6],0.1,0.5)),
    }

def calc_hv(F_min):
    """HV normalizado para [0,1] — comparavel entre runs."""
    gmin=F_min.min(0); gmax=F_min.max(0)
    rng=np.where(gmax-gmin<1e-12,1e-12,gmax-gmin)
    F_norm=(F_min-gmin)/rng
    return float(HV(ref_point=np.ones(F_min.shape[1])*1.1).do(F_norm)), F_norm

def eval_on_test(model_state, feat_dim, backbone_idx):
    """Avalia modelo salvo no cache sobre o conjunto de TESTE."""
    _,_,_,_,f_te,l_te = ALL_FEATURES[backbone_idx]
    te_ld = DataLoader(TensorDataset(f_te,l_te), 256, shuffle=False)
    model = LinearHead(feat_dim, NUM_CLASSES).to(device)
    model.load_state_dict({k:v.to(device) for k,v in model_state.items()})
    model.eval(); preds,labs,probs=[],[],[]
    with torch.no_grad():
        for X,y in te_ld:
            out=model(X.to(device))
            preds.extend(torch.argmax(out,1).cpu().numpy())
            labs.extend(y.numpy())
            probs.extend(torch.softmax(out,1).cpu().numpy())
    preds=np.array(preds); labs=np.array(labs); probs=np.array(probs)
    acc  = float(accuracy_score(labs,preds))
    f1m  = float(f1_score(labs,preds,average='macro',zero_division=0))
    try:    auroc=float(roc_auc_score(labs,probs,multi_class='ovr',average='macro'))
    except: auroc=0.5
    del model; gc.collect()
    return {'accuracy':acc,'f1':f1m,'auroc':auroc,
            'preds':preds,'labels':labs,'probs':probs}

print('Espaco de busca OK')
print(f'  7 variaveis: lr, optimizer, epochs, wd, backbone, batch_size, dropout')
print(f'  Estimativa pop=20 gen=20: ~{(20+19*10)*0.25:.0f} min = {(20+19*10)*0.25/60:.1f} h')


In [ ]:
# [5] Sistemas FIS
U=np.arange(0,1.01,0.01)
def gauss(u,c,s=0.15): return {lbl:fuzz.gaussmf(u,v,s) for lbl,v in c.items()}
def amfs(var,d):
    for lbl,arr in d.items(): var[lbl]=arr
def safe(sim,inputs,fb):
    try:
        for k,v in inputs.items(): sim.input[k]=float(np.clip(v,0.01,0.99))
        sim.compute()
        r=sim.output[list(sim.ctrl.consequents)[0].label]
        return float(np.clip(r,0,1)) if np.isfinite(r) else float(fb)
    except: return float(np.clip(fb,0,1))

OUT5=gauss(U,{'very_low':0.1,'low':0.3,'medium':0.5,'high':0.75,'very_high':0.9},0.12)

def make_fis2(n1,n2,mfs1,mfs2):
    a1=ctrl.Antecedent(U,n1); a2=ctrl.Antecedent(U,n2)
    co=ctrl.Consequent(U,'fitness',defuzzify_method='centroid')
    amfs(a1,gauss(U,mfs1)); amfs(a2,gauss(U,mfs2)); amfs(co,OUT5)
    pairs=[(a1['high'],a2['high'],co['very_high']),(a1['high'],a2['medium'],co['high']),
           (a1['medium'],a2['high'],co['high']),(a1['medium'],a2['medium'],co['medium']),
           (a1['high'],a2['low'],co['medium']),(a1['low'],a2['high'],co['medium']),
           (a1['medium'],a2['low'],co['low']),(a1['low'],a2['medium'],co['low']),
           (a1['low'],a2['low'],co['very_low'])]
    return ctrl.ControlSystemSimulation(ctrl.ControlSystem([ctrl.Rule(h&l,c) for h,l,c in pairs]))

SIM_EVAL=make_fis2('accuracy','f1',
    {'low':0.2,'medium':0.5,'high':0.8},{'low':0.2,'medium':0.5,'high':0.8})
SIM_OOD =make_fis2('auroc','fpr_inv',
    {'low':0.55,'medium':0.75,'high':0.90},{'low':0.2,'medium':0.5,'high':0.8})
SIM_PERF=make_fis2('lat_inv','mem_inv',
    {'low':0.2,'medium':0.5,'high':0.8},{'low':0.2,'medium':0.5,'high':0.8})

ei=ctrl.Antecedent(U,'eval_score'); oi=ctrl.Antecedent(U,'ood_score')
pi=ctrl.Antecedent(U,'perf_score'); qc=ctrl.Consequent(U,'quality',defuzzify_method='centroid')
for v in [ei,oi,pi]: amfs(v,gauss(U,{'low':0.2,'medium':0.5,'high':0.8}))
amfs(qc,gauss(U,{'poor':0.15,'acceptable':0.45,'good':0.70,'excellent':0.90},0.12))
SIM_SEL=ctrl.ControlSystemSimulation(ctrl.ControlSystem([
    ctrl.Rule(ei['high']&oi['high']&pi['high'],   qc['excellent']),
    ctrl.Rule(ei['high']&oi['high']&pi['medium'], qc['excellent']),
    ctrl.Rule(ei['high']&oi['high']&pi['low'],    qc['good']),
    ctrl.Rule(ei['high']&oi['medium']&pi['high'], qc['good']),
    ctrl.Rule(ei['high']&oi['medium'],             qc['good']),
    ctrl.Rule(ei['high']&oi['low'],                qc['acceptable']),
    ctrl.Rule(ei['medium']&oi['high'],             qc['good']),
    ctrl.Rule(ei['medium']&oi['medium']&pi['high'],qc['acceptable']),
    ctrl.Rule(ei['medium']&oi['medium'],           qc['acceptable']),
    ctrl.Rule(ei['medium']&oi['low'],              qc['poor']),
    ctrl.Rule(ei['low'],                           qc['poor']),
]))

print('FIS OK — sanity check:')
for nm,sim,hi,lo in [
    ('FIS-Eval',SIM_EVAL,{'accuracy':0.9,'f1':0.9},{'accuracy':0.2,'f1':0.2}),
    ('FIS-OOD', SIM_OOD, {'auroc':0.9,'fpr_inv':0.9},{'auroc':0.6,'fpr_inv':0.2}),
    ('FIS-Perf',SIM_PERF,{'lat_inv':0.9,'mem_inv':0.9},{'lat_inv':0.2,'mem_inv':0.2}),
]:
    sh=safe(sim,hi,0.5); sl=safe(sim,lo,0.5)
    print(f"  {'OK' if sh>sl else 'FALHOU'}  {nm}  high={sh:.3f}  low={sl:.3f}")


In [ ]:
# [6] Funcao de avaliacao FIS
# Usa xkey(x) para acessar o cache — sem bug de float
def eval_fis(x_vec):
    m    = fast_eval(x_vec)  # retorna do cache ou treina
    mn   = m['backbone']
    lat_n = LATENCY_REF[mn]/LAT_MAX
    mem_n = MEMORY_REF[mn]/MEM_MAX
    fe = safe(SIM_EVAL,{'accuracy':m['accuracy'],'f1':m['f1']},
              0.4*m['accuracy']+0.6*m['f1'])
    fo = safe(SIM_OOD, {'auroc':m['auroc'],'fpr_inv':1-m['fpr']},
              0.5*m['auroc']+0.5*m['accuracy'])
    fp = safe(SIM_PERF,{'lat_inv':1-lat_n,'mem_inv':1-mem_n},
              0.5*(1-lat_n)+0.5*(1-mem_n))
    return float(fe), float(fo), float(fp)

class FISProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=N_VAR,n_obj=3,n_constr=0,xl=xl,xu=xu)
        self.gen=0; self.n_calls=0
    def _evaluate(self,x,out,*args,**kwargs):
        self.n_calls+=1
        fe,fo,fp=eval_fis(x); out['F']=np.array([-fe,-fo,-fp])

print('FISProblem OK')


In [ ]:
# [7] Execucao
POP, GEN = 20, 20

EVAL_CACHE.clear()
prob_f=FISProblem()
alg_f=NSGA2(pop_size=POP,sampling=FloatRandomSampling(),
            crossover=SBX(prob=0.9,eta=15),mutation=PM(eta=20),eliminate_duplicates=True)
hist_f={'gen':[],'fe':[],'fo':[],'fp':[],'nds':[],'hv':[]}

class CbF(Callback):
    def __init__(self): super().__init__(); self.t0=time.time()
    def notify(self,algorithm):
        prob_f.gen=algorithm.n_gen
        Fn=algorithm.opt.get('F'); Fp=-Fn
        hv_now,_=calc_hv(Fn)
        hist_f['gen'].append(algorithm.n_gen)
        hist_f['fe'].append(float(Fp[:,0].max()))
        hist_f['fo'].append(float(Fp[:,1].max()))
        hist_f['fp'].append(float(Fp[:,2].max()))
        hist_f['nds'].append(len(Fn)); hist_f['hv'].append(hv_now)
        t=(time.time()-self.t0)/60
        eta=t/algorithm.n_gen*(GEN-algorithm.n_gen) if algorithm.n_gen>0 else 0
        print(f"  Gen {algorithm.n_gen:2d}/{GEN}  NDS={len(Fn):2d}"
              f"  Eval={Fp[:,0].max():.3f}  HV={hv_now:.4f}"
              f"  [{t:.1f}min ETA:{eta:.0f}min]",flush=True)

print(f'NSGA-II+FIS  pop={POP}  gen={GEN}  dataset={DATASET_NAME}')
res_f=minimize(prob_f,alg_f,get_termination('n_gen',GEN),
               callback=CbF(),seed=SEED,save_history=True,verbose=False)

pF_f=-res_f.F; pX_f=res_f.X
hv_final,_=calc_hv(res_f.F)
print(f'\nPareto FIS: {len(pF_f)} solucoes  |  HV={hv_final:.6f}')
print(f'Reais: {len(EVAL_CACHE)}/{prob_f.n_calls}')


In [ ]:
# [8] FIS-Selecao — ranquear fronteira de Pareto
qual=[]
for row in pF_f:
    sim=ctrl.ControlSystemSimulation(SIM_SEL.ctrl)
    q=safe(sim,{'eval_score':row[0],'ood_score':row[1],'perf_score':row[2]},
           0.5*row[0]+0.3*row[1]+0.2*row[2])
    qual.append(q)
qual=np.array(qual); best_i=int(np.argmax(qual))

# Recuperar metricas pelo xkey — sem bug de float
best_key_f = xkey(pX_f[best_i])
best_m_f   = EVAL_CACHE.get(best_key_f, {})
best_Ff    = pF_f[best_i]

print('FIS-SELECAO — Top 5')
for rk,idx in enumerate(np.argsort(qual)[::-1][:5],1):
    k=xkey(pX_f[idx]); m=EVAL_CACHE.get(k,{})
    print(f"  {rk}. q={qual[idx]:.4f}"
          f"  Eval={pF_f[idx,0]:.3f} OOD={pF_f[idx,1]:.3f} Perf={pF_f[idx,2]:.3f}"
          f"  {m.get('backbone','?')}"
          f"  acc={m.get('accuracy',0)*100:.1f}%")

print(f'\nMelhor: #{best_i+1}  q={qual[best_i]:.4f}  HV={hv_final:.6f}')
print(f'  Backbone : {best_m_f.get("backbone","?")}')
print(f'  Val acc  : {best_m_f.get("accuracy",0)*100:.2f}%')
print(f'  F1 macro : {best_m_f.get("f1",0)*100:.2f}%')
print(f'  AUROC    : {best_m_f.get("auroc",0):.4f}')


In [ ]:
# [9] Avaliacao final no TESTE
print('Avaliando melhor configuracao no conjunto de teste...')
test_m_f = eval_on_test(
    best_m_f['model_state'],
    best_m_f['feat_dim'],
    best_m_f['backbone_idx']
)
print(f'\nRESULTADO FINAL — TESTE')
print(f'  Dataset   : {DATASET_NAME.upper()}')
print(f'  Backbone  : {best_m_f["backbone"]}')
print(f'  Accuracy  : {test_m_f["accuracy"]*100:.2f}%')
print(f'  F1 macro  : {test_m_f["f1"]*100:.2f}%')
print(f'  AUROC     : {test_m_f["auroc"]:.4f}')
print(f'  FIS qual  : {qual[best_i]:.4f}')
print(f'  HV final  : {hv_final:.6f}')
print()
print(classification_report(
    test_m_f['labels'], test_m_f['preds'],
    target_names=CLASSES[:NUM_CLASSES], zero_division=0
))
torch.save({
    'params':      best_m_f.get('params',{}),
    'backbone':    best_m_f.get('backbone',''),
    'model_state': best_m_f.get('model_state',{}),
    'val_metrics': {k:best_m_f.get(k,0) for k in ['accuracy','f1','auroc']},
    'test_metrics':test_m_f,
    'hv':          hv_final,
    'fis_quality': float(qual[best_i]),
    'dataset':     DATASET_NAME,
}, 'best_model_fis.pth')
print('best_model_fis.pth salvo')


In [ ]:
# [10] Visualizacoes
plt.style.use('seaborn-v0_8-darkgrid')
fig,axes=plt.subplots(2,3,figsize=(20,11))
gn=hist_f['gen']
for m,c,l in [('fe','#2ecc71','FIS-Eval'),('fo','#3498db','FIS-OOD'),('fp','#e74c3c','FIS-Perf')]:
    axes[0,0].plot(gn,hist_f[m],'-o',color=c,lw=2,ms=4,label=l)
axes[0,0].set_xlabel('Geracao'); axes[0,0].set_ylabel('FIS Score')
axes[0,0].set_title('Evolucao FIS',fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
axes[0,1].plot(gn,hist_f['hv'],'darkorange',lw=2.5,marker='D',ms=5)
axes[0,1].fill_between(gn,0,hist_f['hv'],alpha=0.15,color='darkorange')
axes[0,1].set_xlabel('Geracao'); axes[0,1].set_ylabel('Hipervolume')
axes[0,1].set_title(f'HV (final={hv_final:.4f})',fontweight='bold'); axes[0,1].grid(alpha=0.3)
sc=axes[0,2].scatter(pF_f[:,0],pF_f[:,1],c=qual,cmap='RdYlGn',s=120,edgecolors='k',vmin=0,vmax=1)
axes[0,2].scatter(best_Ff[0],best_Ff[1],s=300,marker='*',color='gold',edgecolors='k',zorder=5,
                  label=f'q={qual[best_i]:.3f}')
plt.colorbar(sc,ax=axes[0,2],label='quality'); axes[0,2].set_xlabel('FIS-Eval')
axes[0,2].set_ylabel('FIS-OOD'); axes[0,2].set_title('Eval x OOD',fontweight='bold')
axes[0,2].legend(); axes[0,2].grid(alpha=0.3)
av=np.linspace(0.01,0.99,20); fv=np.linspace(0.01,0.99,20); Z=np.zeros((20,20))
for i,a in enumerate(av):
    for j,f in enumerate(fv): Z[i,j]=safe(SIM_EVAL,{'accuracy':a,'f1':f},(a+f)/2)
im=axes[1,0].contourf(fv,av,Z,levels=15,cmap='RdYlGn'); plt.colorbar(im,ax=axes[1,0])
axes[1,0].set_xlabel('F1'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].set_title('Superficie FIS-Eval',fontweight='bold')
cm=confusion_matrix(test_m_f['labels'],test_m_f['preds'])
im2=axes[1,1].imshow(cm,cmap='Blues'); plt.colorbar(im2,ax=axes[1,1])
axes[1,1].set_xlabel('Predito'); axes[1,1].set_ylabel('Real')
axes[1,1].set_title(f'Matriz de Confusao (Teste)',fontweight='bold')
mc={}
for X in pX_f:
    k=xkey(X); bn=EVAL_CACHE.get(k,{}).get('backbone','?'); mc[bn]=mc.get(bn,0)+1
axes[1,2].bar(mc.keys(),mc.values(),color='darkorange',edgecolor='k',alpha=0.85)
axes[1,2].set_ylabel('Solucoes'); axes[1,2].set_title('Backbones no Pareto',fontweight='bold')
axes[1,2].tick_params(axis='x',rotation=15); axes[1,2].grid(alpha=0.3,axis='y')
plt.suptitle(
    f'{DATASET_NAME.upper()} | FIS | '
    f'Teste:{test_m_f["accuracy"]*100:.2f}% F1:{test_m_f["f1"]*100:.2f}% '
    f'| HV={hv_final:.4f} | q={qual[best_i]:.4f}',
    fontsize=11,fontweight='bold')
plt.tight_layout(); plt.savefig('fis_results.png',dpi=300,bbox_inches='tight')
plt.show(); print('fis_results.png salvo')
